In [2]:
import os
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_io as tfio
import numpy as np


def load_protocol(protocol_path):
    df = pd.read_csv(protocol_path, sep=" ", header=None)
    df.columns = ["speaker_id", "file_id", "dash", "system_id", "key"]
    df["label"] = df["key"].map({"bonafide": 0, "spoof": 1})
    return df[["file_id", "label"]]


2025-12-15 03:51:35.464821: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-15 03:51:36.205164: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-15 03:51:38.264997: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow_hub/__init__.py:61: U

In [4]:
train_df = load_protocol("../LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt")
dev_df   = load_protocol("../LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt")
eval_df   = load_protocol("../LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt")


In [5]:
eval_df.head()

,file_id,label
0,LA_E_2834763,1
1,LA_E_8877452,1
2,LA_E_6828287,1
3,LA_E_6977360,1
4,LA_E_5932896,1


In [3]:
y_train = train_df["label"].values.astype("float32")
y_dev   = dev_df["label"].values.astype("float32")
y_eval   = eval_df["label"].values.astype("float32")

In [12]:
from transformers import BertTokenizer, TFBertModel

# Load BERT
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = TFBertModel.from_pretrained("bert-base-uncased", use_safetensors=False)

def get_text_embedding(label, system_id):
    """
    label: 0 (bonafide) or 1 (spoof)
    system_id: string like "A07", ignored for bonafide cases
    """

    if label == 0:
        text = "This is genuine human speech."
    else:
        text = f"This audio is spoofed and generated by attack system {system_id}."

    # BERT tokenization
    tokens = tokenizer(
        text, return_tensors="tf",
        padding=True, truncation=True, max_length=32
    )

    # Forward pass
    outputs = bert(**tokens)

    # Use pooled_output (CLS embedding)
    return outputs.pooler_output[0].numpy()   # shape: (768,)


Some layers from the model checkpoint at bert-base-uncased were not used when initializing TFBertModel: ['nsp___cls', 'mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at bert-base-uncased.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


In [13]:
def extract_text_dataset(df):
    X_text = []

    for _, row in df.iterrows():
        emb = get_text_embedding(row["label"], row["system_id"])
        X_text.append(emb)

    return np.array(X_text, dtype="float32")


In [14]:
X_text_train = extract_text_dataset(train_df)
X_text_dev   = extract_text_dataset(dev_df)
#X_text_eval = 


2025-11-24 11:42:26.369050: W tensorflow/core/framework/op_kernel.cc:1842] UNKNOWN: KeyError: 0
Traceback (most recent call last):

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/data/ops/from_generator_op.py", line 290, in finalize_py_func
    generator_state.iterator_completed(iterator_id)

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/data/ops/dataset_ops.py", line 872, in iterator_completed
    del self._iterators[self._normalize_id(iterator_id)]
        ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyError: 0


2025-11-24 11:42:26.369170:

KeyError: 'system_id'

In [6]:
from tensorflow.keras.models import load_model
model = load_model("fusion_model.keras")

/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
eval_df = load_protocol("LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt")

X_audio_eval = extract_audio_dataset(eval_df, "LA/ASVspoof2019_LA_eval/flac")
X_text_eval  = extract_text_dataset(eval_df)

preds = model.predict({"audio_input": X_audio_eval, "text_input": X_text_eval})


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# y_true: list/array of 0/1 labels
# y_pred: list/array of 0/1 predicted labels

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)
print("Confusion Matrix:\n", cm)


In [1]:
import os
import time
import numpy as np
import soundfile as sf
import tensorflow as tf
import tensorflow_hub as hub
from tqdm import tqdm

# Load YAMNet once
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

# -------------------------------
# Audio loader (memory-friendly)
# -------------------------------
def load_flac_soundfile(path, target_sr=16000):
    """
    Load a FLAC file as mono float32 tensor, resampled to target_sr.
    """
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)  # convert to mono
    wav = tf.convert_to_tensor(wav, dtype=tf.float32)
    
    if sr != target_sr:
        wav = tf.audio.resample(wav, sr, target_sr)
    
    return wav

# -------------------------------
# YAMNet embedding extractor
# -------------------------------
def get_yamnet_embedding(path):
    wav = load_flac_soundfile(path)
    scores, embeddings, _ = yamnet(wav)
    # mean pooling over time frames -> 1024-D
    return tf.reduce_mean(embeddings, axis=0).numpy()

# -------------------------------
# Batch extractor with memmap, tqdm, ETA, and checkpoints
# -------------------------------
def extract_audio_dataset_memmap(df, base_path, save_path, embedding_dim=1024, checkpoint_every=500):
    """
    Extract audio embeddings for a dataset and save to memmap incrementally.
    
    Args:
        df: DataFrame with column "file_id" containing file names without extension
        base_path: path to folder containing FLAC files
        save_path: path to memmap file (e.g., "X_audio_train.dat")
        embedding_dim: YAMNet embedding dimension (default 1024)
        checkpoint_every: flush memmap to disk every N files
    Returns:
        np.memmap object containing all embeddings
    """
    n_files = len(df)
    
    # Create memmap array
    X = np.memmap(save_path, dtype="float32", mode="w+", shape=(n_files, embedding_dim))
    
    start_time = time.time()
    
    for i, fid in enumerate(tqdm(df["file_id"], desc="Extracting audio embeddings")):
        path = os.path.join(base_path, f"{fid}.flac")
        
        # Skip files that fail to load
        try:
            emb = get_yamnet_embedding(path)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            emb = np.zeros(embedding_dim, dtype="float32")
        
        X[i] = emb
        
        # Checkpoint memmap to disk every N files
        if (i + 1) % checkpoint_every == 0 or (i + 1) == n_files:
            X.flush()
            
            elapsed = time.time() - start_time
            avg_time = elapsed / (i + 1)
            remaining = avg_time * (n_files - (i + 1))
            print(f"Checkpoint saved at {i+1}/{n_files} files | Elapsed: {elapsed:.1f}s | ETA: {remaining/60:.1f} min")
    
    return X


2025-11-29 13:34:50.419255: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-29 13:34:51.046728: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-29 13:34:52.783586: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow_hub/__init__.py:61: U

In [6]:
train_base_path = "../LA/ASVspoof2019_LA_train/flac"
dev_base_path   = "../LA/ASVspoof2019_LA_dev/flac"
eval_base_path = "../LA/ASVspoof2019_LA_eval/flac"


#X_audio_train = extract_audio_dataset_memmap(train_df, train_base_path, "X_audio_train.dat")
X_audio_dev   = extract_audio_dataset_memmap(dev_df, dev_base_path, "X_audio_dev.dat")
#X_audio_eval   = extract_audio_dataset_memmap(dev_df, dev_base_path, "X_audio_eval.dat")



Extracting audio embeddings:   2%|█                                                | 539/24844 [00:04<02:04, 194.81it/s]

Checkpoint saved at 500/24844 files | Elapsed: 4.4s | ETA: 3.6 min


Extracting audio embeddings:   4%|█▉                                              | 1019/24844 [00:07<02:08, 184.72it/s]

Checkpoint saved at 1000/24844 files | Elapsed: 7.0s | ETA: 2.8 min


Extracting audio embeddings:   6%|██▉                                             | 1526/24844 [00:10<02:21, 165.32it/s]

...

Checkpoint saved at 24000/24844 files | Elapsed: 148.3s | ETA: 0.1 min


Extracting audio embeddings:  99%|██████████████████████████████████████████████▍| 24529/24844 [02:31<00:02, 147.33it/s]

Checkpoint saved at 24500/24844 files | Elapsed: 151.6s | ETA: 0.0 min


Extracting audio embeddings: 100%|███████████████████████████████████████████████| 24844/24844 [02:33<00:00, 161.46it/s]

Checkpoint saved at 24844/24844 files | Elapsed: 153.9s | ETA: 0.0 min


In [7]:
import tensorflow as tf
import numpy as np

def dataset_generator(X_audio_path, X_text, y, batch_size=16):
    """
    Yield batches from memmap audio + in-memory text + labels.
    """
    n_samples = len(y)
    
    # Open memmap
    X_audio = np.memmap(X_audio_path, dtype="float32", mode="r", shape=(n_samples, 1024))
    
    # Shuffle indices
    indices = np.arange(n_samples)
    
    while True:
        np.random.shuffle(indices)
        
        for start in range(0, n_samples, batch_size):
            end = min(start + batch_size, n_samples)
            batch_idx = indices[start:end]
            
            batch_audio = X_audio[batch_idx]
            batch_text  = X_text[batch_idx]
            batch_y     = y[batch_idx]
            
            yield {"audio_input": batch_audio, "text_input": batch_text}, batch_y


In [2]:
batch_size = 16

train_dataset = tf.data.Dataset.from_generator(
    lambda: dataset_generator("X_audio_train.dat", X_text_train, y_train, batch_size),
    output_signature=(
        {
            "audio_input": tf.TensorSpec(shape=(None, 1024), dtype=tf.float32),
            "text_input": tf.TensorSpec(shape=(None, 768), dtype=tf.float32)
        },
        tf.TensorSpec(shape=(None, 1), dtype=tf.int32)
    )
).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: dataset_generator("X_audio_dev.dat", X_text_dev, y_dev, batch_size),
    output_signature=(
        {
            "audio_input": tf.TensorSpec(shape=(None, 1024), dtype=tf.float32),
            "text_input": tf.TensorSpec(shape=(None, 768), dtype=tf.float32)
        },
        tf.TensorSpec(shape=(None, 1), dtype=tf.int32)
    )
).prefetch(tf.data.AUTOTUNE)


I0000 00:00:1764002242.985963     736 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3539 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [11]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    steps_per_epoch=len(y_train) // batch_size,
    validation_steps=len(y_dev) // batch_size
)


Epoch 1/20


2025-11-24 11:40:19.075689: W tensorflow/core/framework/op_kernel.cc:1842] UNKNOWN: NameError: name 'X_text_train' is not defined
Traceback (most recent call last):

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/data/ops/dataset_ops.py", line 865, in get_iterator
    return self._iterators[iterator_id]
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^

KeyError: 0


During handling of the above exception, another exception occurred:


Traceback (most recent call last):

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/data/ops/from_generator_op.py", line 198, 

UnknownError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) UNKNOWN:  NameError: name 'X_text_train' is not defined
Traceback (most recent call last):

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/data/ops/dataset_ops.py", line 865, in get_iterator
    return self._iterators[iterator_id]
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^

KeyError: 0


During handling of the above exception, another exception occurred:


Traceback (most recent call last):

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kw

In [3]:
print(tf.__version__)

2.20.0
